
# ML Assignment 2 - Model Training Notebook (.py format)<br>
<br>
**Course:** Machine Learning (M.Tech AIML) - Assignment 2<br>
<br>
This notebook-style script does the full modelling<br>
pipeline required by the assignment:<br>
<br>
1. loads the dataset (Breast Cancer Wisconsin - Diagnostic, a UCI dataset),<br>
2. splits it into train/test and writes `train_data.csv` + `test_data.csv`,<br>
3. trains the 5 required classifiers,<br>
4. evaluates all 6 required metrics (Accuracy, AUC, Precision, Recall, F1, MCC),<br>
5. prints the comparison table + per-model observations, and<br>
6. saves every trained model to `model/*.pkl` and the metrics to `model/metrics.json`.<br>
<br>
Running this script regenerates ALL artefacts the repository / Streamlit app need,<br>
so it is the single source of truth for the results shown in the README and app.

------------------------------------------------------------------<br>
Imports. Only standard scientific-Python + scikit-learn are used, matching<br>
requirements.txt (no deep-learning frameworks are needed for this task).<br>
------------------------------------------------------------------

In [ ]:
import json
import pathlib
import numpy as np
import pandas as pd

In [ ]:
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

The 5 required models

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.ensemble import RandomForestClassifier

Evaluation metrics

In [ ]:
from sklearn.metrics import (accuracy_score, roc_auc_score, precision_score,
                             recall_score, f1_score, matthews_corrcoef,
                             confusion_matrix, classification_report)
import joblib

In [ ]:
RANDOM_STATE = 42
HERE = pathlib.Path(__file__).resolve().parent if "__file__" in globals() else pathlib.Path(".").resolve()
ROOT = HERE.parent if HERE.name == "model" else HERE      # repo root
MODEL_DIR = ROOT / "model"
MODEL_DIR.mkdir(parents=True, exist_ok=True)
print("Repo root :", ROOT)
print("Model dir :", MODEL_DIR)

## Step 1 - Dataset<br>
**Breast Cancer Wisconsin (Diagnostic)** is a classic UCI classification dataset:<br>
569 instances and 30 real-valued features (mean/SE/worst of 10 nuclear<br>
measurements such as radius, texture, perimeter, area, ...). It satisfies the<br>
assignment minimums (>= 12 features, >= 500 instances) and is a binary problem.<br>
<br>
**Label convention:** scikit-learn ships the target as 0 = malignant, 1 = benign.<br>
We RELABEL so that the positive class (1) = malignant (cancer). This makes<br>
Precision / Recall directly interpretable for cancer screening (recall = the<br>
fraction of true cancers we catch), which is the clinically important quantity.

In [ ]:
raw = load_breast_cancer(as_frame=True)
feature_names = list(raw.feature_names)

In [ ]:
X = raw.frame[feature_names].copy()
y = (raw.target.values == 0).astype(int)          # 1 = malignant, 0 = benign
y = pd.Series(y, name="target")

In [ ]:
n_instances, n_features = X.shape
print(f"Instances: {n_instances}  |  Features: {n_features}  |  Classes: {sorted(y.unique())}")
print(f"Positive (malignant=1) count: {int(y.sum())}  |  Negative (benign=0) count: {int((1-y).sum())}")
assert n_features >= 12 and n_instances >= 500, "Dataset does not meet the assignment minimums!"

## Step 2 - Train/test split and CSV export<br>
<br>
Stratified 80/20 split (keeps the malignant/benign ratio in both parts). We save<br>
BOTH parts to CSV: `test_data.csv` is what users upload to the Streamlit app;<br>
`train_data.csv` lets the app retrain the models if the saved `.pkl` files are<br>
unavailable (e.g. a scikit-learn version mismatch on the cloud).

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=RANDOM_STATE)

Persist the splits (features + target column) as CSVs in the repo root.

In [ ]:
train_df = X_train.copy(); train_df["target"] = y_train.values
test_df  = X_test.copy();  test_df["target"]  = y_test.values
train_df.to_csv(ROOT / "train_data.csv", index=False)
test_df.to_csv(ROOT / "test_data.csv", index=False)
print(f"Wrote train_data.csv ({train_df.shape[0]} rows) and test_data.csv ({test_df.shape[0]} rows)")

## Step 3 - Define the 5 models<br>
<br>
Distance-/gradient-based models (Logistic Regression, kNN) are wrapped in a<br>
`Pipeline` with `StandardScaler`, because they are sensitive to feature scale.<br>
Tree-based models (Decision Tree, Random Forest) and Gaussian Naive Bayes are<br>
scale-invariant, so they are used directly. Bundling the scaler INTO the pipeline<br>
means the saved model already knows how to preprocess raw input - the app can<br>
feed it the raw CSV with no extra steps.

In [ ]:
def make_models():
    """Return a fresh dict {display_name: unfitted estimator/pipeline}."""
    return {
        "Logistic Regression": Pipeline([
            ("scaler", StandardScaler()),
            ("clf", LogisticRegression(max_iter=5000, random_state=RANDOM_STATE)),
        ]),
        "Decision Tree": DecisionTreeClassifier(random_state=RANDOM_STATE),
        "kNN": Pipeline([
            ("scaler", StandardScaler()),
            ("clf", KNeighborsClassifier(n_neighbors=5)),
        ]),
        "Naive Bayes": GaussianNB(),                      # Gaussian NB (continuous features)
        "Random Forest": RandomForestClassifier(
            n_estimators=200, random_state=RANDOM_STATE),
    }

Map display names -> safe filenames for the saved .pkl models.

In [ ]:
FILE_KEYS = {
    "Logistic Regression": "logistic_regression",
    "Decision Tree": "decision_tree",
    "kNN": "knn",
    "Naive Bayes": "naive_bayes",
    "Random Forest": "random_forest",
}

## Step 4 - Train, evaluate (6 metrics), and save<br>
<br>
For every model we compute the six required metrics on the held-out test set:<br>
Accuracy, AUC (from predicted probabilities of the positive class), Precision,<br>
Recall, F1 and Matthews Correlation Coefficient (MCC). Each trained model is<br>
saved with `joblib`.

In [ ]:
def evaluate(model, X_te, y_te):
    """Fit-independent evaluation: assumes `model` is already fitted."""
    y_pred = model.predict(X_te)
    # Probability of the positive class (malignant=1) for the AUC / ROC.
    if hasattr(model, "predict_proba"):
        y_score = model.predict_proba(X_te)[:, 1]
    else:                                   # safety fallback (not needed for these 5)
        y_score = model.decision_function(X_te)
    return {
        "Accuracy":  accuracy_score(y_te, y_pred),
        "AUC":       roc_auc_score(y_te, y_score),
        "Precision": precision_score(y_te, y_pred, zero_division=0),
        "Recall":    recall_score(y_te, y_pred, zero_division=0),
        "F1":        f1_score(y_te, y_pred, zero_division=0),
        "MCC":       matthews_corrcoef(y_te, y_pred),
    }

In [ ]:
results = {}
models = make_models()
for name, model in models.items():
    model.fit(X_train, y_train)                          # train
    results[name] = evaluate(model, X_test, y_test)      # score
    joblib.dump(model, MODEL_DIR / f"{FILE_KEYS[name]}.pkl")   # persist
    print(f"trained + saved: {name:20s} -> model/{FILE_KEYS[name]}.pkl")

## Step 5 - Comparison table (the table that goes into the README)

In [ ]:
metrics_df = pd.DataFrame(results).T[["Accuracy", "AUC", "Precision", "Recall", "F1", "MCC"]]
metrics_df = metrics_df.round(4)
print("\n================= MODEL COMPARISON (test set) =================")
print(metrics_df.to_string())

Save the metrics so the README / app can display identical numbers.

In [ ]:
(MODEL_DIR / "metrics.json").write_text(json.dumps(
    {k: {m: float(v) for m, v in d.items()} for k, d in results.items()}, indent=2))
print("\nSaved metrics to model/metrics.json")

Identify the overall winner using F1 (a balanced choice for this medical task,<br>
where both false negatives and false positives matter; MCC agrees below).

In [ ]:
winner = metrics_df["F1"].idxmax()
print(f"\nOverall winner by F1-score: {winner}")
print("Ranking by MCC:")
print(metrics_df["MCC"].sort_values(ascending=False).to_string())

## Step 6 - Confusion matrix + classification report for the winning model<br>
(Printed here for the write-up; the Streamlit app shows this interactively for<br>
whichever model the user selects.)

In [ ]:
best_model = models[winner]
best_pred = best_model.predict(X_test)
print(f"\nConfusion matrix - {winner} (rows=true, cols=pred; classes [benign=0, malignant=1]):")
print(confusion_matrix(y_test, best_pred))
print(f"\nClassification report - {winner}:")
print(classification_report(y_test, best_pred, target_names=["benign (0)", "malignant (1)"]))

## Observations (also summarised in the README)<br>
<br>
(These are printed from the computed numbers so they always match the run.)

In [ ]:
obs_order = metrics_df["MCC"].sort_values(ascending=False)
print("Per-model observations (data-driven):")
for name in results:
    r = results[name]
    print(f"- {name}: accuracy {r['Accuracy']:.3f}, AUC {r['AUC']:.3f}, "
          f"recall {r['Recall']:.3f}, F1 {r['F1']:.3f}, MCC {r['MCC']:.3f}")
print(f"\nBest overall: {winner} (highest F1); AUC ranking is led by "
      f"{metrics_df['AUC'].idxmax()}.")
print("Done.")